## Imports and loading

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import MinMaxScaler

from sklearn.impute import SimpleImputer
import html
import re

print("Caricamento dati...")

train_df = pd.read_csv("development.csv")
train_df['title'] = train_df['title'].fillna('')
train_df['article'] = train_df['article'].fillna('')
train_df['text_content'] = train_df['title']+ ' ' + train_df['article']


eval_df = pd.read_csv("evaluation.csv")

eval_df['title'] = eval_df['title'].fillna('')
eval_df['article'] = eval_df['article'].fillna('')
eval_df['text_content'] = eval_df['title']+ ' '  + eval_df['article']

print("Dati caricati.")
print(train_df[train_df['page_rank']==5].shape)

import re
import html

def clean_text(text):
    if not isinstance(text, str):
        return "" # Gestione robusta dei NaN

    # 1. Decodifica entità HTML (es. &quot; -> ")
    text = html.unescape(text)
    
    # 2. Rimozione Tag HTML (es. <br>, <p>, <div>)
    #text = re.sub(r'<[^>]+>', ' ', text)
    
    # 3. Rimozione URL e Email
    #text = re.sub(r'http\S+', ' ', text)  
    #text = re.sub(r'www\.\S+', ' ', text)
    #text = re.sub(r'\S+@\S+', ' ', text)
    
    # 4. Rimozione Metadata editoriali tra parentesi, rimuove "(Reuters)", "(AP)", "(CNN)".
    # Evita che il modello impari la fonte invece del contenuto.
    #text = re.sub(r'\([A-Za-z\s\.]+\)', ' ', text)
    
    # 6. Rimozione caratteri singoli isolati
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    
    # 7. Normalizzazione Spazi e Lowercase
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    
    return text

train_df['text_content'] = train_df['text_content'].apply(clean_text)
eval_df['text_content'] = eval_df['text_content'].apply(clean_text)

Caricamento dati...
Dati caricati.
(73891, 8)


In [13]:
columns_to_check = [col for col in train_df.columns if col != 'Id']

# Identificazione dei duplicati
duplicates = train_df.duplicated(subset=columns_to_check, keep='first')
num_duplicates = duplicates.sum()

print(f"Numero di righe duplicate rilevate: {num_duplicates}")

# Visualizzazione di alcuni esempi di righe duplicate
if num_duplicates > 0:
    print(train_df[duplicates].head())

train_df.drop_duplicates(subset=columns_to_check, keep='first', inplace=True)

Numero di righe duplicate rilevate: 334
          Id   source                                             title  \
7152    7152  Reuters  Bin Laden Alive, Releases Audio Tape -- Web Site   
9130    9130      BBC                 Shining Path retrial halted again   
10085  10085      BBC                  US renews commitment to Pakistan   
10436  10436   Forbes       ADV: $175,000 Mortgage for Under $730/Month   
10841  10841      CNN       ADV: $150,000 Mortgage for Under $690/Month   

                                                 article  page_rank  \
7152   An audio recording purportedly by Osama bin La...          5   
9130   The retrial of Peruvian guerrilla leader Abima...          5   
10085  US envoy Richard Armitage pledges new backing ...          5   
10436  Mortgage rates are at record lows. Save $1000s...          5   
10841  Mortgage rates are at record lows. Save $1000s...          5   

                 timestamp  label  \
7152   0000-00-00 00:00:00      0   
9130   0

## Class for top_n_sources

In [14]:
from sklearn.base import BaseEstimator, TransformerMixin

class TopNSourcesImputer(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=50, other_label='other'):
        self.top_n = top_n
        self.other_label = other_label
        self.top_sources_ = []

    def fit(self, X, y=None):
        """
        Calcola le top_n fonti basandosi solo sui dati di training passati.
        X ci si aspetta sia un DataFrame o una Series pandas (o array numpy).
        """
        # Convertiamo in Series per usare value_counts facilmente
        if isinstance(X, np.ndarray):
            X = pd.Series(X.ravel())
        elif isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]
            
        # Calcoliamo la lista delle top N
        self.top_sources_ = X.value_counts().nlargest(self.top_n).index.tolist()
        return self

    def transform(self, X):
        """
        Sostituisce le fonti non in top_sources_ con 'other'
        """
        # Assicuriamoci di lavorare su una copia per non modificare i dati originali
        X_copy = X.copy()
        
        # Gestione input (Pandas vs Numpy)
        if isinstance(X_copy, pd.DataFrame):
            # Applichiamo la trasformazione sulla colonna
            return X_copy.map(lambda x: x if x in self.top_sources_ else self.other_label)
        elif isinstance(X_copy, pd.Series):
             return X_copy.apply(lambda x: x if x in self.top_sources_ else self.other_label).to_frame()
        else:
            # Numpy array
            vfunc = np.vectorize(lambda x: x if x in self.top_sources_ else self.other_label)
            return vfunc(X_copy).reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return input_features

## Preprocessing

In [15]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder


text_transformer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=3,
    max_df=0.5
    )
#X=text_transformer.fit_transform(train_df['text_content'])
#print(f"Number of features from text: {X.shape[1]}")
cat_transformer = OneHotEncoder(handle_unknown='ignore')
num_transformer = MinMaxScaler()
hour_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ])
source_pipe = Pipeline([
    ('imputer_top_n', TopNSourcesImputer(top_n=50)), # Default iniziale a 50
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])
ord_transformer = Pipeline([
    ('enc', OrdinalEncoder(categories=[[2,3,4,5]])),
    ('scaler', MinMaxScaler())])
preprocessor = ColumnTransformer(
    transformers=[
        ('txt', text_transformer, 'text_content'),
        ('ord', cat_transformer, ['page_rank']),
        ('hour', hour_transformer, ['hour_cos','hour_sin']),
        ('cat_source', source_pipe, ['source'])
        #('day', day_transformer, ['day_of_week']),
        #('num', num_transformer, ['text_lenght']),
        
    ]
)
train_df['day_of_week'] = pd.to_datetime(train_df['timestamp'], errors='coerce').dt.dayofweek
eval_df['day_of_week'] = pd.to_datetime(eval_df['timestamp'], errors='coerce').dt.dayofweek

train_df['is_weekend'] = train_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
train_df['timestamp_missing'] = train_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)
eval_df['is_weekend'] = eval_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
eval_df['timestamp_missing'] = eval_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)

train_df['text_lenght'] = train_df['text_content'].apply(lambda x: len(x))
eval_df['text_lenght'] = eval_df['text_content'].apply(lambda x: len(x))
train_df['hour_of_day'] = pd.to_datetime(train_df['timestamp'], errors='coerce').dt.hour
eval_df['hour_of_day'] = pd.to_datetime(eval_df['timestamp'], errors='coerce').dt.hour
train_df['hour_sin'] = np.sin(2 * np.pi * train_df['hour_of_day'] / 24)
train_df['hour_cos'] = np.cos(2 * np.pi * train_df['hour_of_day'] / 24)

eval_df['hour_sin'] = np.sin(2 * np.pi * eval_df['hour_of_day'] / 24)
eval_df['hour_cos'] = np.cos(2 * np.pi * eval_df['hour_of_day'] / 24)

X_eval = eval_df[['text_content', 'source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]


X = train_df[['text_content','source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]
y = train_df['label']


## Linear SVC

In [16]:

 ## LINEAR SVC MODEL

classifier = LinearSVC()
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])


param_grid = {
    'clf__C': [0.18],
    'clf__class_weight': ['balanced'],
    'clf__loss':['squared_hinge'],
    'preprocessor__txt__ngram_range': [(1,3)],
    'preprocessor__cat_source__imputer_top_n__top_n': [300] # ora è fissato a 300, poi mettiamo più valori

}

grid_search = GridSearchCV(
    model_pipeline, 
    param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=2
)

grid_search.fit(X, y)
print("LINEAR SVC - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_



Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV] END clf__C=0.18, clf__class_weight=balanced, clf__loss=squared_hinge, preprocessor__cat_source__imputer_top_n__top_n=300, preprocessor__txt__ngram_range=(1, 3); total time=  20.2s
[CV] END clf__C=0.18, clf__class_weight=balanced, clf__loss=squared_hinge, preprocessor__cat_source__imputer_top_n__top_n=300, preprocessor__txt__ngram_range=(1, 3); total time=  20.2s
[CV] END clf__C=0.18, clf__class_weight=balanced, clf__loss=squared_hinge, preprocessor__cat_source__imputer_top_n__top_n=300, preprocessor__txt__ngram_range=(1, 3); total time=  20.3s
[CV] END clf__C=0.18, clf__class_weight=balanced, clf__loss=squared_hinge, preprocessor__cat_source__imputer_top_n__top_n=300, preprocessor__txt__ngram_range=(1, 3); total time=  20.3s
[CV] END clf__C=0.18, clf__class_weight=balanced, clf__loss=squared_hinge, preprocessor__cat_source__imputer_top_n__top_n=300, preprocessor__txt__ngram_range=(1, 3); total time=  20.3s
LINEAR SVC - Gri

In [19]:
best_model.fit(X, y)
y_eval_pred = best_model.predict(X_eval)

pd.DataFrame({
    'Id': eval_df['Id'],
    'Predicted': y_eval_pred}).to_csv('submission.csv', index=False)

## Logistic Regression

In [18]:

## LOGISTIC REGRESSION MODEL


from sklearn.linear_model import LogisticRegression

classifier=LogisticRegression()

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])



param_grid = {
    'clf__C': [1],
    'clf__class_weight': ['balanced'],
    'clf__max_iter': [120]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2)

grid_search.fit(X, y)

print("LOGISTIC REGRESSION - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")


Fitting 5 folds for each of 1 candidates, totalling 5 fits


KeyboardInterrupt: 

In [ ]:
from sklearn.naive_bayes import MultinomialNB

classifier=MultinomialNB()

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])



param_grid = {
    'clf__alpha': [0.1],
    'clf__fit_prior': [True]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2)

grid_search.fit(X, y)

print("Multinomial Naive Bayes - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")



Fitting 5 folds for each of 1 candidates, totalling 5 fits


KeyboardInterrupt: 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier=RandomForestClassifier()

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])



param_grid = {
    'clf__n_estimators': [200],      
    'clf__max_depth': [None],     
    'clf__min_samples_split': [ 5],     
    'clf__class_weight': ['balanced']
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2)

grid_search.fit(X, y)

print("RANDOM FOREST - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")


Fitting 5 folds for each of 1 candidates, totalling 5 fits
RANDOM FOREST - Grid Search Results:
{'clf__class_weight': 'balanced', 'clf__max_depth': None, 'clf__min_samples_split': 5, 'clf__n_estimators': 200}
Best Cross-Validation Macro F1: 0.7032


In [ ]:


from sklearn.neighbors import KNeighborsClassifier


classifier=KNeighborsClassifier()

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])



param_grid = {
    'clf__n_neighbors': [100],
    'clf__weights': ['uniform', 'distance'],
    'clf__metric' : ['euclidean', 'manhattan', 'cosine']
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2)

grid_search.fit(X, y)

print("KNN - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")



Fitting 3 folds for each of 6 candidates, totalling 18 fits
KNN - Grid Search Results:
{'clf__metric': 'cosine', 'clf__n_neighbors': 100, 'clf__weights': 'distance'}
Best Cross-Validation Macro F1: 0.6567
